# Phase 1 — Baseline Validation (B0): `whisper-base` on LibriSpeech dev-clean / dev-other

**Policy role.** Per the research policy's approval flow, Phase 1 reproduces the baseline under the
**frozen configuration of EXPERIMENT_SPEC.md §6**, and *"another team member reviews the implementation
and independently verifies the result. The project cannot proceed until the baseline is approved."*
This notebook is therefore written for two audiences: the implementer (first run) and the **reviewer**
(independent re-run) — the final section specifies the verification protocol and pass criteria.

**What B0 is (and is not).** B0 is plain `whisper-base` beam-search decoding with **no biasing**, under
the *same* frozen decoding parameters that B1 and P will use later — including `use_cache=False`,
which is unusual for a baseline but required by §6 so that no condition differs in anything except the
bias term. Consequence: our B0 WER is the **project's internal reference number**, not a reproduction
of the Whisper paper's exact figures (different normalizer, different decoding heuristics); the
published numbers serve only as a sanity corridor (Section 6).

**Reproducibility mechanics built in:**
- every frozen parameter echoed into `phase1_config.json` together with package versions and the
  resolved model revision;
- transcription writes an **append-only JSONL per split** and is resumable — rerunning the notebook
  skips finished utterances (a full run is ~5,500 utterances; interruptions are expected);
- TUNE/DEVTEST shard assignment is a pure deterministic function of the utterance-ID list and seed 42
  (sharding happens **before** the ≤29 s length filter, so the shard rule is independent of audio
  decoding — the diagnostic notebook uses the identical rule on text alone);
- SHA-256 checksums over shard membership and transcripts for the Phase 5 audit trail.

**Runtime.** Full splits (~2,700 + ~2,900 utterances) with beam 5 and no KV cache: several hours on
GPU. `SUBSET_PER_SPLIT` exists for smoke runs (Phase 3 uses it); **approval-grade numbers require
`SUBSET_PER_SPLIT = None`.**


## Section 0.1 — Dependencies

In [ ]:
# Run once, then restart the kernel.
# %pip install -U torch transformers datasets jiwer soundfile librosa pandas


## Section 0.2 — Imports, seed, device, version capture

Line-by-line: the usual imports plus `hashlib`/`platform` for the audit trail. `VERSIONS` is captured
*now* and lands in the config artifact — the reviewer compares theirs against it before comparing WER
(a version mismatch explains small deltas before anyone suspects the implementation).


In [ ]:
import os, json, random, re, hashlib, platform
import numpy as np
import torch
import pandas as pd
import jiwer
import transformers, datasets as datasets_lib

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

if torch.cuda.is_available():
    DEVICE = "cuda"
elif torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"

VERSIONS = {
    "python": platform.python_version(),
    "platform": platform.platform(),
    "torch": torch.__version__,
    "transformers": transformers.__version__,
    "datasets": datasets_lib.__version__,
    "jiwer": jiwer.__version__,
    "device": DEVICE,
}
print(json.dumps(VERSIONS, indent=2))


## Section 0.3 — The frozen configuration (spec §6, verbatim)

Every entry below is **locked by the approved spec**; changing any of them invalidates the run under
Rule 1. `SUBSET_PER_SPLIT` is the one deliberate exception — it exists so Phase 3 can smoke-test the
identical code path; the artifact records its value so a subset run can never masquerade as the full
baseline.


In [ ]:
FROZEN = {
    "model": "openai/whisper-base",
    "num_beams": 5,
    "use_cache": False,          # parity with condition P (spec §6) — applies to ALL conditions
    "language": "en",
    "task": "transcribe",
    "timestamps": False,
    "max_audio_s": 29.0,
    "sampling_rate": 16000,
    "seed": SEED,
    "tune_frac": 0.25,
    "normalization": "lowercase; keep [a-z' ]; collapse whitespace",
}
SUBSET_PER_SPLIT = None          # None = full split (approval-grade). Small int = smoke run only.

SPLITS = {"dev-clean": ("clean", "validation"), "dev-other": ("other", "validation")}

def norm(text):
    text = text.lower()
    text = re.sub(r"[^a-z' ]+", " ", text)
    return re.sub(r"\s+", " ", text).strip()

print("Frozen config loaded. SUBSET_PER_SPLIT =", SUBSET_PER_SPLIT)


## Section 1 — Model under the frozen config

Line-by-line:
1. Load processor + model, `.eval()`, pinned decoding prompt (`<|en|><|transcribe|><|notimestamps|>`).
2. `MODEL_REVISION` — the resolved HF commit hash of the checkpoint actually downloaded; recorded so
   the reviewer can verify they decoded with byte-identical weights.
3. `transcribe_b0()` — the entire baseline: feature-extract → `generate` with **only** frozen
   parameters → decode. Deliberately minimal; B1/P will differ from this function by one
   `logits_processor` argument and nothing else.


In [ ]:
from transformers import WhisperProcessor, WhisperForConditionalGeneration

processor = WhisperProcessor.from_pretrained(FROZEN["model"])
whisper = (WhisperForConditionalGeneration
           .from_pretrained(FROZEN["model"]).to(DEVICE).eval())
tok = processor.tokenizer
tok.set_prefix_tokens(language="english", task="transcribe")
MODEL_REVISION = getattr(whisper.config, "_commit_hash", "unknown")
print("model revision:", MODEL_REVISION)

@torch.no_grad()
def transcribe_b0(audio):
    feats = processor.feature_extractor(
        audio, sampling_rate=FROZEN["sampling_rate"], return_tensors="pt"
    ).input_features.to(DEVICE)
    ids = whisper.generate(feats,
                           language=FROZEN["language"], task=FROZEN["task"],
                           num_beams=FROZEN["num_beams"], use_cache=FROZEN["use_cache"])
    return tok.decode(ids[0], skip_special_tokens=True).strip()


## Section 2 — Resumable transcription pass over both splits

How the loop stays reviewer-friendly, line-by-line:
1. **Resume state:** already-transcribed IDs are read back from the split's JSONL; length-excluded IDs
   from its exclusion file. A rerun touches neither — it continues where the last run stopped, and a
   completed run becomes a fast no-op that just recollects the ID list.
2. **Streaming:** utterances arrive lazily from the Hub in a fixed order; nothing near the full corpus
   is stored. `all_ids` collects every utterance ID *before* any filtering — the shard rule's input.
3. **Length filter:** audio longer than 29 s is recorded in `phase1_excluded_<split>.json` with its
   duration (spec §3 requires the exclusion count).
4. Each new transcription is appended and flushed immediately — a crash costs at most one utterance.
5. `get_audio()` absorbs `datasets`-version differences in audio decoding, as in earlier notebooks.


In [ ]:
from datasets import load_dataset, Audio

def get_audio(sample):
    a = sample["audio"]
    if isinstance(a, dict) and a.get("array") is not None:
        return np.asarray(a["array"], dtype=np.float32)
    if hasattr(a, "get_all_samples"):
        return a.get_all_samples().data.numpy().astype(np.float32).flatten()
    import soundfile as sf
    return sf.read(a["path"], dtype="float32")[0]

def run_split(split_name):
    hf_config, hf_split = SPLITS[split_name]
    tpath = f"phase1_transcripts_{split_name}.jsonl"
    xpath = f"phase1_excluded_{split_name}.json"
    done = set()
    if os.path.exists(tpath):
        with open(tpath) as f:
            done = {json.loads(l)["id"] for l in f if l.strip()}
    excluded = json.load(open(xpath)) if os.path.exists(xpath) else {}

    stream = load_dataset("openslr/librispeech_asr", hf_config,
                          split=hf_split, streaming=True)
    stream = stream.cast_column("audio", Audio(sampling_rate=FROZEN["sampling_rate"]))

    all_ids, n_new = [], 0
    out = open(tpath, "a")
    for s in stream:
        if SUBSET_PER_SPLIT is not None and len(all_ids) >= SUBSET_PER_SPLIT:
            break
        uid = s["id"]
        all_ids.append(uid)
        if uid in done or uid in excluded:
            continue
        audio = get_audio(s)
        dur = len(audio) / FROZEN["sampling_rate"]
        if dur > FROZEN["max_audio_s"]:
            excluded[uid] = round(dur, 2)
            continue
        out.write(json.dumps({"id": uid, "ref": s["text"],
                              "hyp": transcribe_b0(audio)}) + "\n")
        out.flush()
        n_new += 1
        if n_new % 25 == 0:
            print(f"  {split_name}: {len(done) + n_new} transcribed "
                  f"({len(all_ids)} seen, {len(excluded)} excluded)")
    out.close()
    json.dump(excluded, open(xpath, "w"))
    json.dump(all_ids, open(f"phase1_ids_{split_name}.json", "w"))
    print(f"{split_name}: {len(all_ids)} utterances seen, "
          f"{len(done) + n_new} transcribed, {len(excluded)} excluded (>29 s)")
    return all_ids

ALL_IDS = {}
for split_name in SPLITS:
    print(f"=== {split_name} ===")
    ALL_IDS[split_name] = run_split(split_name)


## Section 3 — Deterministic TUNE / DEVTEST shards

The rule (spec §3), line-by-line: **sort** the full ID list (removes any dependence on stream order),
shuffle with `random.Random(42)` (a fresh, isolated RNG — global state cannot perturb it), first 25% →
TUNE, rest → DEVTEST. Sharding uses IDs only — no audio, no filtering — so the Phase 0 diagnostic
notebook reproduces identical shards from text alone. Membership lists and their SHA-256 go into the
artifacts; the reviewer's first check is that their checksums match, which verifies the entire data
pipeline in one comparison.


In [ ]:
def shard_ids(ids, tune_frac, seed):
    ordered = sorted(ids)
    random.Random(seed).shuffle(ordered)
    k = int(len(ordered) * tune_frac)
    return set(ordered[:k]), set(ordered[k:])

def sha(obj):
    return hashlib.sha256(json.dumps(sorted(obj)).encode()).hexdigest()[:16]

SHARDS = {}
for split_name, ids in ALL_IDS.items():
    tune, devtest = shard_ids(ids, FROZEN["tune_frac"], FROZEN["seed"])
    SHARDS[split_name] = {"TUNE": tune, "DEVTEST": devtest}
    json.dump({"TUNE": sorted(tune), "DEVTEST": sorted(devtest)},
              open(f"phase1_shards_{split_name}.json", "w"))
    print(f"{split_name}: TUNE={len(tune)} (sha {sha(tune)})  "
          f"DEVTEST={len(devtest)} (sha {sha(devtest)})")


## Section 4 — B0 WER (the reference numbers)

Computed with the frozen normalization, per split × {ALL, TUNE, DEVTEST}. The shard rows matter later:
B1/P tuning reads TUNE, reporting reads DEVTEST — this table is where their baselines come from.


In [ ]:
rows = []
RECORDS = {}
for split_name in SPLITS:
    with open(f"phase1_transcripts_{split_name}.jsonl") as f:
        recs = [json.loads(l) for l in f if l.strip()]
    RECORDS[split_name] = recs
    for shard_name, idset in [("ALL", None)] + list(SHARDS[split_name].items()):
        sel = [r for r in recs if idset is None or r["id"] in idset]
        wer = jiwer.wer([norm(r["ref"]) for r in sel],
                        [norm(r["hyp"]) for r in sel])
        rows.append({"split": split_name, "shard": shard_name,
                     "n_utts": len(sel), "wer": round(wer, 4)})

results = pd.DataFrame(rows)
results


## Section 5 — Sanity corridor vs. published results

`whisper-base` (multilingual, 74 M) is commonly reported around **~5% WER on dev/test-clean** and
**~12% on dev/test-other** *under Whisper's own text normalizer and decoding heuristics*. Our frozen
config differs on both counts (stricter normalizer; plain beam-5, no temperature fallback), so treat
**4–8% (dev-clean)** and **10–16% (dev-other)** as the coarse sanity corridor, and fill
`PUBLISHED_REFERENCE` with the exact figures from the Whisper paper's appendix tables (or the model
card) when citing. A result outside the corridor fails Phase 1 regardless of internal consistency —
it means the pipeline, not the model, is wrong (resampling, normalization, or decoding bug).


In [ ]:
PUBLISHED_REFERENCE = {          # fill from Radford et al. (2022) appendix / HF model card
    "dev-clean": None,
    "dev-other": None,
}
CORRIDOR = {"dev-clean": (0.04, 0.08), "dev-other": (0.10, 0.16)}

for split_name in SPLITS:
    wer = float(results[(results["split"] == split_name)
                        & (results["shard"] == "ALL")]["wer"].iloc[0])
    lo, hi = CORRIDOR[split_name]
    status = "WITHIN sanity corridor" if lo <= wer <= hi else "OUTSIDE corridor — INVESTIGATE"
    print(f"{split_name}: B0 WER = {wer:.4f}  [{lo:.2f}, {hi:.2f}] → {status}")


## Section 6 — Audit artifacts

`phase1_config.json` is the single file the reviewer needs to reproduce everything: frozen parameters,
package versions, model revision, subset flag, per-split counts, shard checksums, transcript-file
checksums, and the results table. Transcript JSONLs and shard files are committed alongside.


In [ ]:
def file_sha(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()[:16]

artifact = {
    "phase": "1-baseline-validation",
    "frozen_config": FROZEN,
    "subset_per_split": SUBSET_PER_SPLIT,
    "versions": VERSIONS,
    "model_revision": MODEL_REVISION,
    "splits": {
        s: {
            "n_seen": len(ALL_IDS[s]),
            "n_transcribed": len(RECORDS[s]),
            "shard_sha": {k: sha(v) for k, v in SHARDS[s].items()},
            "transcripts_sha": file_sha(f"phase1_transcripts_{s}.jsonl"),
        } for s in SPLITS
    },
    "results": rows,
}
json.dump(artifact, open("phase1_config.json", "w"), indent=2)
results.to_csv("phase1_results.csv", index=False)
print(json.dumps(artifact["splits"], indent=2))
print("\nSaved: phase1_config.json, phase1_results.csv, "
      "phase1_transcripts_*.jsonl, phase1_shards_*.json, phase1_excluded_*.json")


## Section 7 — Reviewer verification protocol (Rule 2)

To the assigned reviewer — independent verification means, concretely:

1. **Fresh clone, fresh environment.** Install the dependency cell's packages; compare your version
   printout against `phase1_config.json → versions`. Note any differences before running.
2. **Run every cell top-to-bottom** with `SUBSET_PER_SPLIT = None` (delete any `phase1_*.jsonl` you
   did not produce yourself — resume must not launder the implementer's transcripts into your run).
3. **Compare, in order:**
   - shard SHAs — must match **exactly** (pure function of the ID list and seed; a mismatch means a
     data-pipeline difference and stops the review);
   - per-split utterance counts and exclusion counts — must match exactly;
   - WER table — **identical hardware/versions:** match to 4 decimals (beam search here is
     deterministic); **different hardware:** floating-point divergence can flip rare beam ties —
     tolerate |ΔWER| ≤ 0.002 absolute per split, and flag anything larger as a finding;
   - sanity corridor — both splits inside.
4. **Record the verdict** (approve / findings) with your own `phase1_config.json` attached.

**Phase 1 pass condition:** reviewer-reproduced numbers within tolerance, corridors satisfied,
verdict recorded. Only then does the spec unfreeze Phase 2 (B1/P implementation).
